In [ ]:
import numpy as np
import bacco
import matplotlib.pyplot as plt

import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
def pars(i, mstar):

    arr = np.vstack( ( mstar, np.ones(len(mstar)) * wind_en[i],\
                        np.ones(len(mstar)) * wind_vel[i],\
                        np.ones(len(mstar)) * rho_rec[i],\
                        np.ones(len(mstar)) * sf_ts[i],\
                        np.ones(len(mstar)) * ef_kin[i],\
                        np.ones(len(mstar)) * ef_high[i],\
                        np.ones(len(mstar)) * f_re[i])).T

    return arr

In [ ]:
wind_en_or      = []
wind_vel_or     = []
rho_rec_or      = []
sf_ts_or        = []
ef_kin_or       = []
ef_high_or      = []
f_re_or         = []

for i in range(31):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    else:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"

    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en_or.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel_or.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec_or.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin_or.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re_or.append(float(line.split()[1]))

rho_rec_or = np.log10(rho_rec_or)
ef_kin_or = np.log10(ef_kin_or)
        
wind_en   = (np.asarray(wind_en_or) - np.mean(wind_en_or)) / np.std(wind_en_or)
wind_vel  = (np.asarray(wind_vel_or) - np.mean(wind_vel_or)) / np.std(wind_vel_or)
rho_rec   = (np.asarray(rho_rec_or) - np.mean(rho_rec_or)) / np.std(rho_rec_or)
sf_ts     = (np.asarray(sf_ts_or) - np.mean(sf_ts_or)) / np.std(sf_ts_or)
ef_kin    = (np.asarray(ef_kin_or) - np.mean(ef_kin_or)) / np.std(ef_kin_or)
ef_high   = (np.asarray(ef_high_or) - np.mean(ef_high_or)) / np.std(ef_high_or)
f_re      = (np.asarray(f_re_or) - np.mean(f_re_or)) / np.std(f_re_or)

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME
    
_snap = 264

zoom = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(_snap,_snap), dm_file="snapdir_{:03d}/snapshot_{:03d}".format(_snap,_snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, use_ids=False, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(_snap,_snap), numpart=4320**3)



In [ ]:
# Load halo selection
with open("/cosmos_storage/home/fgmaion/MTNG-resims/halo_selection/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []
    for line in f.readlines():
        final_sel.append(int(line.split()[0]))
final_sel = np.array(final_sel)

In [ ]:
xmatch = utils.cross_match(zoom, snap=264, name='fiducial')

# Load MTNG and get the fraction of halos to do the upweighting
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

zoom_split = utils.split_halos(zoom)

zoom_sel = {}
zoom_sel['sel'] = xmatch['ind'][:,np.newaxis,np.newaxis]
zoom_sel['h_frac'] = h_frac[np.newaxis, :]

In [ ]:
zoom_split = utils.split_halos(zoom)
mtng_split = utils.split_halos(mtng)

In [ ]:
result = zoom_split.rhalf_m2half(sel_mask=zoom_sel, nbins=10)

In [ ]:
rhalf_mtng = np.array([[4057837104866.573, 194.96785114902266],
                        [4905537587625.687, 227.71157232045582],
                        [3519706338365.108, 174.1511986082501],
                        [2843157253562.988, 144.96111476841122],
                        [2351719937567.1143, 122.37967098989976],
                        [1727623610489.3428, 92.28754054723241],
                        [1269353262993.8354, 72.60011169302058],
                        [889478260338.7367, 55.52212603592578],
                        [623320561755.8387, 43.06406559304907],
                        [320989469727.399, 27.41060436590089],
                        [102856541555.42126, 11.425273289813495],
                        [58242789969.03958, 8.02715116055984],
                        [48188487631.18631, 7.271431410919081],
                        [34595468126.34626, 6.493559711833273],
                        [22069041261.777588, 5.880348586352421],
                        [13432394310.281496, 5.794540762590753],
                        [6012787165.015008, 5.7888426301984435],
                        [3029561228.894818, 5.949323482091947],
                        [1263352225.8798437, 6.027299125065391],
                        [502606746.23103434, 6.460012237893936],
                        [165543707.37286395, 7.4275247790808905],
                        [100753318.96335945, 7.216721038307759],
                        [62770650.32984774, 6.535029979995543]])


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

ax.scatter(result['m2half'], 1e3 * np.array(result['rhalf']), color='C0', marker='s', s=1)
ax.plot(result['m2half_mean'], 1e3 * result['rhalf_mean'], color='C0')
ax.plot(rhalf_mtng[:,0], rhalf_mtng[:,1], label='MTNG', color="C3")

ax.set_xlabel(r"$M_{2\mathrm{half}}$ [$M_\odot$]")
ax.set_ylabel(r"$\log_{10} R_{1/2}$ [kpc]")

In [ ]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']# + ['bf_sim'] 

In [ ]:
Nbins_rhalf = 9

# Estimate the size-mass relation
zoom_rhalf = {}

for i in range(len(name_list)):
    zoom_rhalf[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/size_mass_rel/rhalf_m2half_{}_Nbins{:d}.npy".format(name_list[i], Nbins_rhalf), allow_pickle=True)[0]


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

for i in range(25, 26):
    ax.plot(zoom_rhalf[name_list[i]]['m2half_mean'], 1e3 * zoom_rhalf[name_list[i]]['rhalf_mean'], color='gray', marker='o')
    ax.scatter(zoom_rhalf[name_list[i]]['m2half'], np.array( zoom_rhalf[name_list[i]]['rhalf'] ) * 1e3, color='gray', marker='s', s=1)

ax.plot(rhalf_mtng[:,0], rhalf_mtng[:,1], label='MTNG', color="C3")

ax.set_xlabel(r"$M_{2\mathrm{half}}$ [$M_\odot$]")
ax.set_ylabel(r"$\log_{10} R_{1/2}$ [kpc]")

## Load the computed relations

In [ ]:
# Load Size-Mass Relation
Nbins_rhalf = 9

zoom_rhalf = {}
for i in range(len(name_list)):
    zoom_rhalf[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/size_mass_rel/rhalf_m2half_{}_Nbins{:d}.npy".format(name_list[i], Nbins_rhalf), allow_pickle=True)[0]


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

for i in range(len(name_list)-1):
    ax.plot(zoom_rhalf[name_list[i]]['m2half_mean'], 1e3 * zoom_rhalf[name_list[i]]['rhalf_mean'], color='gray',
    alpha=0.5, lw=1, marker='o')

ax.plot(zoom_rhalf['fiducial']['m2half_mean'], 1e3 * zoom_rhalf['fiducial']['rhalf_mean'], label='Fiducial', color='C3', lw=2)
ax.plot(rhalf_mtng[:,0], rhalf_mtng[:,1], label='MTNG', color="C0")

ax.set_xlabel(r"$M_{2\mathrm{half}}$ [$M_\odot$]")
ax.set_ylabel(r"$\log_{10} R_{1/2}$ [kpc]")

ax.legend()


In [ ]:
import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/scripts")
from GP_models import SMF_Model, fgas_Model
import torch
import gpytorch

In [ ]:
model_rhalf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_model_size_mass.pth")
likelihood_rhalf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_likelihood_size_mass.pth")

model_rhalf.eval()
likelihood_rhalf.eval()

In [ ]:
# Define the training set
train_sel = np.arange(31)

In [ ]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(2, 3, figsize=(15, 10), dpi=100)
    plt.subplots_adjust(wspace=0.15, hspace=0.3)

    for i in range(2):
        for j in range(3):
            mask = ~np.isnan(zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean']) & ~np.isnan(zoom_rhalf[name_list[train_sel[3*i+j]]]['rhalf_mean'])
            test_x = torch.asarray(pars(train_sel[3*i+j], np.log10(zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean'][mask])), dtype=torch.float)
            observed_pred = likelihood_rhalf(model_rhalf(torch.asarray(test_x, dtype=torch.float)))
        
            ax[i,j].plot(np.log10(rhalf_mtng[:,0]), np.log10(rhalf_mtng[:,1] / 1e3), color="C3", lw=3)

            ax[i,j].set_xlim(8.5,13)
            ax[i,j].set_ylim(-2.5, -1)

            ax[i,j].set_title('Simulation {:s}'.format(name_list[train_sel[3*i+j]]), fontsize=16)

            if j==0:
                ax[i,j].set_ylabel('$\log_{10}(R_{1/2}/\mathrm{Mpc})$', fontsize=16)
            
            ax[i,j].set_xlabel('$\log_{10}(M_{2\mathrm{half}}/M_\odot)$', fontsize=16)

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()

            # Shade between the lower and upper confidence bounds
            ax[i,j].fill_between(test_x[:,0].numpy(), observed_pred.mean.numpy()-observed_pred.stddev.numpy(), observed_pred.mean.numpy()+observed_pred.stddev.numpy(), color='C0', alpha=0.4, edgecolor=None)
            ax[i,j].plot(test_x[:,0], observed_pred.mean.numpy(), 'C0', lw=2, label="GP Prediction", alpha=0.9)

            # Plot test data as blue squares
            ax[i,j].plot(np.log10(zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean'][mask]), np.log10(zoom_rhalf[name_list[train_sel[3*i+j]]]['rhalf_mean'][mask]), 's', color="C0", label="Test Data")

ax[0,0].legend(loc='lower left', fontsize=12)

plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/results/testing_plots/size_mass_test.pdf", bbox_inches='tight')

In [ ]:
zoom_rhalf[name_list[train_sel[3*i+j]]]['m2half_mean']